# Embedding Analysis

This notebook analyzes learned embeddings from trained models using:
- t-SNE visualization
- Cluster quality metrics (silhouette score, inter/intra class distances)
- SNR-stratified analysis

In [ ]:
import sys
from pathlib import Path

src_path = Path("../src")
if src_path.exists():
    sys.path.insert(0, str(src_path.resolve()))

import matplotlib.pyplot as plt
import torch

from robust_amc.data import Compose, PowerNormalize, get_data_loaders
from robust_amc.data.radioml_loader import MODULATION_CLASSES
from robust_amc.data.transforms import ToTensor
from robust_amc.evaluation import (
    get_embeddings,
    compute_cluster_metrics,
    plot_embeddings_tsne,
    plot_embeddings_by_snr,
)
from robust_amc.models import create_pfcnn, create_clsr_amc
from robust_amc.utils import get_device

## 1. Setup

In [ ]:
DATA_PATH = Path("../data/RML2016.10a_dict.pkl")
CHECKPOINTS_DIR = Path("../checkpoints")

device = get_device("auto")
print(f"Using device: {device}")

In [ ]:
# Load data
if DATA_PATH.exists():
    transform = Compose([PowerNormalize(), ToTensor()])
    loaders = get_data_loaders(
        DATA_PATH,
        batch_size=256,
        train_transform=transform,
        eval_transform=transform,
        num_workers=0,
    )
    print(f"Test set: {len(loaders['test'].dataset)} samples")
else:
    print(f"Dataset not found at {DATA_PATH}")

## 2. Load Models

In [ ]:
models = {}

# Baseline
baseline_path = CHECKPOINTS_DIR / "baseline_2016" / "best_model.pt"
if baseline_path.exists():
    model = create_pfcnn(num_classes=len(MODULATION_CLASSES))
    ckpt = torch.load(baseline_path, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    models["Baseline"] = model
    print("Loaded: Baseline")

# MDA-DMC
mda_path = CHECKPOINTS_DIR / "mda_dmc_2016" / "best_model.pt"
if mda_path.exists():
    model = create_pfcnn(num_classes=len(MODULATION_CLASSES))
    ckpt = torch.load(mda_path, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    models["MDA-DMC"] = model
    print("Loaded: MDA-DMC")

# CLSR-AMC
clsr_path = CHECKPOINTS_DIR / "clsr_amc_2016" / "best_model.pt"
if clsr_path.exists():
    model = create_clsr_amc(num_classes=len(MODULATION_CLASSES))
    ckpt = torch.load(clsr_path, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    models["CLSR-AMC"] = model
    print("Loaded: CLSR-AMC")

if not models:
    print("No models found! Run training scripts first.")

## 3. Extract and Analyze Embeddings

In [ ]:
all_metrics = {}

for name, model in models.items():
    print(f"\n{'='*40}")
    print(f"Analyzing: {name}")
    print("="*40)
    
    # Extract embeddings
    print("Extracting embeddings...")
    embeddings, labels, snrs = get_embeddings(model, loaders["test"], device=device)
    print(f"  Shape: {embeddings.shape}")
    
    # Compute metrics
    print("Computing cluster metrics...")
    metrics = compute_cluster_metrics(embeddings, labels)
    all_metrics[name] = metrics
    
    print(f"  Silhouette Score: {metrics['silhouette_score']:.4f}")
    print(f"  Inter/Intra Ratio: {metrics['inter_intra_ratio']:.4f}")

## 4. t-SNE Visualization

In [ ]:
for name, model in models.items():
    embeddings, labels, snrs = get_embeddings(model, loaders["test"], device=device)
    
    fig = plot_embeddings_tsne(
        embeddings, labels, MODULATION_CLASSES,
        title=f"{name} Embeddings (t-SNE)",
        n_samples=3000,
    )
    plt.show()

## 5. SNR-Stratified Analysis

In [ ]:
# Pick one model for detailed SNR analysis
model_name = list(models.keys())[0] if models else None

if model_name:
    model = models[model_name]
    embeddings, labels, snrs = get_embeddings(model, loaders["test"], device=device)
    
    fig = plot_embeddings_by_snr(
        embeddings, labels, snrs, MODULATION_CLASSES,
        title_prefix=model_name,
    )
    plt.show()

## 6. Comparison Summary

In [ ]:
if all_metrics:
    print(f"{'Model':<15} {'Silhouette':>12} {'Inter/Intra':>12}")
    print("-" * 41)
    for name, metrics in all_metrics.items():
        print(f"{name:<15} {metrics['silhouette_score']:>12.4f} {metrics['inter_intra_ratio']:>12.4f}")
    
    print("\nInterpretation:")
    print("  - Higher silhouette score = better cluster separation (range: -1 to 1)")
    print("  - Higher inter/intra ratio = more compact, well-separated clusters")